# 1. Market Based Pricing From Financial Modeling Prep

This notebook estimates an implied share price from market multiples instead of intrinsic cash-flow assumptions. It pulls a peer set from Financial Modeling Prep, collects current market multiples, and converts peer median multiples into implied prices for the selected company.

Change `ticker_str` in `../../params.py` and rerun the notebook. Inputs like `BRK\B`, `BRK/B`, and `BRK.B` are normalized automatically for FMP.

In [ ]:
# 2. Setup and FMP helpers
from pathlib import Path
import os
import runpy

import pandas as pd
import plotly.graph_objects as go
import requests


base_path_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PARAMS_FILE_CANDIDATES = []
for base_path in base_path_candidates:
    PARAMS_FILE_CANDIDATES.extend(
        [
            base_path / "params.py",
            base_path / "Single Asset Profile" / "params.py",
            base_path / "Notebooks" / "Single Asset Profile" / "params.py",
            base_path / "Investment Scripts" / "Notebooks" / "Single Asset Profile" / "params.py",
        ]
    )
for params_file in PARAMS_FILE_CANDIDATES:
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
single_asset_params = notebook_params["get_single_asset_profile_params"]()

TICKER = single_asset_params["ticker_str"]  # Shared Single Asset Profile ticker.
BASE_URL = "https://financialmodelingprep.com/stable"


def find_project_root(start_path: Path | None = None) -> Path:
    current = (start_path or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    return current


def load_project_env() -> None:
    env_path = find_project_root() / ".env"
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if value and value[0] == value[-1] and value[0] in {'"', "'"}:
            value = value[1:-1]
        os.environ.setdefault(key, value)


def get_api_key() -> str:
    load_project_env()
    api_key = os.getenv("FMP_API_KEY") or os.getenv("FINANCIAL_MODELING_PREP_API_KEY")
    if api_key:
        return api_key
    raise KeyError("Missing FMP_API_KEY in the project .env file or environment.")


def normalize_symbol(ticker: str) -> str:
    normalized = str(ticker).strip().upper()
    for old, new in ((chr(92), "."), ("/", "."), ("-", ".")):
        normalized = normalized.replace(old, new)
    return normalized


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return payload


def safe_first_record(payload):
    if isinstance(payload, list):
        return payload[0] if payload else {}
    if isinstance(payload, dict):
        return payload
    return {}


def pick_numeric(record, *keys):
    for key in keys:
        value = record.get(key)
        numeric_value = pd.to_numeric(value, errors="coerce")
        if pd.notna(numeric_value):
            return float(numeric_value)
    return pd.NA


def apply_standard_figure_layout(figure: go.Figure, title: str, yaxis_title: str) -> None:
    figure.update_layout(
        title=title,
        template="plotly_dark",
        paper_bgcolor="#020817",
        plot_bgcolor="#0f172a",
        font={"color": "#e2e8f0"},
        hovermode="x unified",
        hoverlabel={"bgcolor": "#0f172a", "font_color": "#e2e8f0"},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
        xaxis_title="Method",
        yaxis_title=yaxis_title,
        autosize=True,
        height=680,
        margin={"l": 60, "r": 30, "t": 90, "b": 60},
    )
    figure.update_xaxes(showgrid=False, automargin=True)
    figure.update_yaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True)


SYMBOL = normalize_symbol(TICKER)
SYMBOL

In [ ]:
# 3. Load company context and market inputs
quote_snapshot = safe_first_record(request_json("quote", symbol=SYMBOL))
profile_snapshot = safe_first_record(request_json("profile", symbol=SYMBOL))
ratio_snapshot = safe_first_record(request_json("ratios-ttm", symbol=SYMBOL))
key_metrics_snapshot = safe_first_record(request_json("key-metrics-ttm", symbol=SYMBOL))

company_name = quote_snapshot.get("name") or profile_snapshot.get("companyName") or SYMBOL
chart_label = f"{company_name} ({SYMBOL})" if company_name != SYMBOL else SYMBOL
current_price = pick_numeric(quote_snapshot, "price")
market_cap = pick_numeric(quote_snapshot, "marketCap", "marketCapTTM")
share_count = pick_numeric(
    key_metrics_snapshot,
    "weightedAverageShsOutDilTTM",
    "weightedAverageShsOutDil",
    "weightedAverageShsOutTTM",
    "weightedAverageShsOut",
    "sharesOutstanding",
)
if pd.isna(share_count):
    share_count = pick_numeric(quote_snapshot, "sharesOutstanding", "sharesNumber")

net_debt = pick_numeric(key_metrics_snapshot, "netDebtTTM", "netDebt")
if pd.isna(net_debt):
    net_debt = 0.0

current_pe = pick_numeric(ratio_snapshot, "priceToEarningsRatioTTM", "peRatioTTM")
current_ps = pick_numeric(ratio_snapshot, "priceToSalesRatioTTM", "priceToSalesRatio")
current_pb = pick_numeric(ratio_snapshot, "priceToBookRatioTTM", "priceToBookRatio")
current_pfcf = pick_numeric(
    ratio_snapshot,
    "priceToFreeCashFlowsRatioTTM",
    "priceToFreeCashFlowsRatio",
)

eps_ttm = pick_numeric(key_metrics_snapshot, "netIncomePerShareTTM", "netIncomePerShare")
if pd.isna(eps_ttm) and pd.notna(current_price) and pd.notna(current_pe) and current_pe > 0:
    eps_ttm = current_price / current_pe

revenue_per_share_ttm = pick_numeric(key_metrics_snapshot, "revenuePerShareTTM", "revenuePerShare")
if pd.isna(revenue_per_share_ttm) and pd.notna(current_price) and pd.notna(current_ps) and current_ps > 0:
    revenue_per_share_ttm = current_price / current_ps

book_value_per_share_ttm = pick_numeric(key_metrics_snapshot, "bookValuePerShareTTM", "bookValuePerShare")
if pd.isna(book_value_per_share_ttm) and pd.notna(current_price) and pd.notna(current_pb) and current_pb > 0:
    book_value_per_share_ttm = current_price / current_pb

free_cash_flow_per_share_ttm = pick_numeric(
    key_metrics_snapshot,
    "freeCashFlowPerShareTTM",
    "freeCashFlowPerShare",
)
if pd.isna(free_cash_flow_per_share_ttm) and pd.notna(current_price) and pd.notna(current_pfcf) and current_pfcf > 0:
    free_cash_flow_per_share_ttm = current_price / current_pfcf

revenue_ttm = pick_numeric(key_metrics_snapshot, "revenueTTM")
if pd.isna(revenue_ttm):
    if pd.notna(revenue_per_share_ttm) and pd.notna(share_count):
        revenue_ttm = revenue_per_share_ttm * share_count
    elif pd.notna(market_cap) and pd.notna(current_ps) and current_ps > 0:
        revenue_ttm = market_cap / current_ps

enterprise_value = pick_numeric(key_metrics_snapshot, "enterpriseValueTTM", "enterpriseValue")
if pd.isna(enterprise_value) and pd.notna(market_cap):
    enterprise_value = market_cap + net_debt

ebitda_ttm = pick_numeric(key_metrics_snapshot, "ebitdaTTM", "ebitda")
current_ev_to_ebitda = pick_numeric(
    key_metrics_snapshot,
    "enterpriseValueOverEBITDATTM",
    "enterpriseValueOverEBITDA",
)
if pd.isna(ebitda_ttm) and pd.notna(enterprise_value) and pd.notna(current_ev_to_ebitda) and current_ev_to_ebitda > 0:
    ebitda_ttm = enterprise_value / current_ev_to_ebitda

ebit_ttm = pick_numeric(key_metrics_snapshot, "ebitTTM", "ebit")
current_ev_to_ebit = pick_numeric(
    key_metrics_snapshot,
    "enterpriseValueOverEBITTTM",
    "enterpriseValueOverEBIT",
)
if pd.isna(ebit_ttm) and pd.notna(enterprise_value) and pd.notna(current_ev_to_ebit) and current_ev_to_ebit > 0:
    ebit_ttm = enterprise_value / current_ev_to_ebit

company_market_inputs = pd.DataFrame(
    [
        {
            "symbol": SYMBOL,
            "companyName": company_name,
            "sector": profile_snapshot.get("sector"),
            "industry": profile_snapshot.get("industry"),
            "currentPrice": current_price,
            "marketCap": market_cap,
            "shareCount": share_count,
            "netDebt": net_debt,
            "enterpriseValue": enterprise_value,
            "revenueTtm": revenue_ttm,
            "ebitdaTtm": ebitda_ttm,
            "ebitTtm": ebit_ttm,
            "epsTtm": eps_ttm,
            "revenuePerShareTtm": revenue_per_share_ttm,
            "bookValuePerShareTtm": book_value_per_share_ttm,
            "freeCashFlowPerShareTtm": free_cash_flow_per_share_ttm,
            "currentPe": current_pe,
            "currentPs": current_ps,
            "currentPb": current_pb,
            "currentPfcf": current_pfcf,
            "currentEvToRevenue": pick_numeric(
                key_metrics_snapshot,
                "enterpriseValueOverRevenueTTM",
                "enterpriseValueOverRevenue",
            ),
            "currentEvToEbitda": current_ev_to_ebitda,
            "currentEvToEbit": current_ev_to_ebit,
        }
    ]
)
company_market_inputs

## 4. Peer Universe

This block uses FMP's `stock-peers` endpoint as the starting universe, then applies only light screening so mega-cap names still retain a workable comparable set. If the stricter pass yields too few candidates, the notebook falls back to a broader selection from the raw peer list.

In [ ]:
# 5. Select a market-based peer set
max_peer_candidates = 25
target_peer_count = 8
require_same_sector = False
require_same_industry = False
market_cap_lower_multiple = 0.05
market_cap_upper_multiple = 20.0

peer_payload = request_json("stock-peers", symbol=SYMBOL)
if isinstance(peer_payload, list):
    if peer_payload and isinstance(peer_payload[0], str):
        raw_peer_symbols = peer_payload
    else:
        raw_peer_symbols = []
        for item in peer_payload:
            if isinstance(item, dict):
                raw_peer_symbols.extend(item.get("peersList", []))
                raw_peer_symbols.extend(item.get("peers", []))
                raw_peer_symbols.extend(item.get("symbols", []))
                if item.get("symbol"):
                    raw_peer_symbols.append(item["symbol"])
                if item.get("peerSymbol"):
                    raw_peer_symbols.append(item["peerSymbol"])
elif isinstance(peer_payload, dict):
    raw_peer_symbols = list(peer_payload.get("peersList", [])) or list(peer_payload.get("peers", []))
else:
    raw_peer_symbols = []

normalized_peer_symbols = []
for peer_symbol in raw_peer_symbols:
    normalized_peer_symbol = normalize_symbol(peer_symbol)
    if normalized_peer_symbol and normalized_peer_symbol != SYMBOL and normalized_peer_symbol not in normalized_peer_symbols:
        normalized_peer_symbols.append(normalized_peer_symbol)

target_sector = profile_snapshot.get("sector")
target_industry = profile_snapshot.get("industry")
target_market_cap = market_cap

peer_screen_rows = []
for peer_symbol in normalized_peer_symbols[:max_peer_candidates]:
    peer_quote = safe_first_record(request_json("quote", symbol=peer_symbol))
    peer_profile = safe_first_record(request_json("profile", symbol=peer_symbol))
    peer_market_cap = pick_numeric(peer_quote, "marketCap", "marketCapTTM")

    same_sector = peer_profile.get("sector") == target_sector if require_same_sector else True
    same_industry = peer_profile.get("industry") == target_industry if require_same_industry else True
    within_market_cap_band = True
    market_cap_distance = pd.NA
    if pd.notna(target_market_cap) and pd.notna(peer_market_cap) and target_market_cap > 0:
        within_market_cap_band = (
            target_market_cap * market_cap_lower_multiple <= peer_market_cap <= target_market_cap * market_cap_upper_multiple
        )
        market_cap_distance = abs(peer_market_cap / target_market_cap - 1)

    peer_screen_rows.append(
        {
            "symbol": peer_symbol,
            "companyName": peer_quote.get("name") or peer_profile.get("companyName") or peer_symbol,
            "sector": peer_profile.get("sector"),
            "industry": peer_profile.get("industry"),
            "marketCap": peer_market_cap,
            "marketCapDistance": market_cap_distance,
            "passesFilters": bool(same_sector and same_industry and within_market_cap_band),
        }
    )

peer_screen = pd.DataFrame(peer_screen_rows)
if not peer_screen.empty:
    peer_screen["marketCapDistance"] = pd.to_numeric(peer_screen["marketCapDistance"], errors="coerce")
    filtered_peer_symbols = peer_screen.loc[peer_screen["passesFilters"], "symbol"].tolist()
    ranked_peer_symbols = peer_screen.sort_values(
        by=["passesFilters", "marketCapDistance", "marketCap"],
        ascending=[False, True, False],
        na_position="last",
    )["symbol"].tolist()
else:
    filtered_peer_symbols = []
    ranked_peer_symbols = []

selected_peers = filtered_peer_symbols[:target_peer_count]
if len(selected_peers) < target_peer_count:
    for peer_symbol in ranked_peer_symbols:
        if peer_symbol not in selected_peers:
            selected_peers.append(peer_symbol)
        if len(selected_peers) >= target_peer_count:
            break

peer_selection_summary = pd.DataFrame(
    [
        {
            "symbol": SYMBOL,
            "rawPeerCount": len(normalized_peer_symbols),
            "filteredPeerCount": len(filtered_peer_symbols),
            "selectedPeerCount": len(selected_peers),
            "sameSectorRequired": require_same_sector,
            "sameIndustryRequired": require_same_industry,
            "marketCapBand": f"{market_cap_lower_multiple:.2f}x to {market_cap_upper_multiple:.1f}x",
        }
    ]
)
peer_selection_summary

In [ ]:
# 6. Collect peer market multiples
peer_metric_rows = []
for peer_symbol in selected_peers:
    peer_quote = safe_first_record(request_json("quote", symbol=peer_symbol))
    peer_profile = safe_first_record(request_json("profile", symbol=peer_symbol))
    peer_ratios = safe_first_record(request_json("ratios-ttm", symbol=peer_symbol))
    peer_key_metrics = safe_first_record(request_json("key-metrics-ttm", symbol=peer_symbol))

    peer_price = pick_numeric(peer_quote, "price")
    peer_market_cap = pick_numeric(peer_quote, "marketCap", "marketCapTTM")
    peer_share_count = pick_numeric(
        peer_key_metrics,
        "weightedAverageShsOutDilTTM",
        "weightedAverageShsOutDil",
        "weightedAverageShsOutTTM",
        "weightedAverageShsOut",
        "sharesOutstanding",
    )
    if pd.isna(peer_share_count):
        peer_share_count = pick_numeric(peer_quote, "sharesOutstanding", "sharesNumber")

    peer_net_debt = pick_numeric(peer_key_metrics, "netDebtTTM", "netDebt")
    if pd.isna(peer_net_debt):
        peer_net_debt = 0.0

    peer_pe = pick_numeric(peer_ratios, "priceToEarningsRatioTTM", "peRatioTTM")
    peer_ps = pick_numeric(peer_ratios, "priceToSalesRatioTTM", "priceToSalesRatio")
    peer_pb = pick_numeric(peer_ratios, "priceToBookRatioTTM", "priceToBookRatio")
    peer_pfcf = pick_numeric(
        peer_ratios,
        "priceToFreeCashFlowsRatioTTM",
        "priceToFreeCashFlowsRatio",
    )

    peer_eps_ttm = pick_numeric(peer_key_metrics, "netIncomePerShareTTM", "netIncomePerShare")
    if pd.isna(peer_eps_ttm) and pd.notna(peer_price) and pd.notna(peer_pe) and peer_pe > 0:
        peer_eps_ttm = peer_price / peer_pe

    peer_revenue_per_share_ttm = pick_numeric(peer_key_metrics, "revenuePerShareTTM", "revenuePerShare")
    if pd.isna(peer_revenue_per_share_ttm) and pd.notna(peer_price) and pd.notna(peer_ps) and peer_ps > 0:
        peer_revenue_per_share_ttm = peer_price / peer_ps

    peer_book_value_per_share_ttm = pick_numeric(peer_key_metrics, "bookValuePerShareTTM", "bookValuePerShare")
    if pd.isna(peer_book_value_per_share_ttm) and pd.notna(peer_price) and pd.notna(peer_pb) and peer_pb > 0:
        peer_book_value_per_share_ttm = peer_price / peer_pb

    peer_free_cash_flow_per_share_ttm = pick_numeric(
        peer_key_metrics,
        "freeCashFlowPerShareTTM",
        "freeCashFlowPerShare",
    )
    if pd.isna(peer_free_cash_flow_per_share_ttm) and pd.notna(peer_price) and pd.notna(peer_pfcf) and peer_pfcf > 0:
        peer_free_cash_flow_per_share_ttm = peer_price / peer_pfcf

    peer_revenue_ttm = pick_numeric(peer_key_metrics, "revenueTTM")
    if pd.isna(peer_revenue_ttm):
        if pd.notna(peer_revenue_per_share_ttm) and pd.notna(peer_share_count):
            peer_revenue_ttm = peer_revenue_per_share_ttm * peer_share_count
        elif pd.notna(peer_market_cap) and pd.notna(peer_ps) and peer_ps > 0:
            peer_revenue_ttm = peer_market_cap / peer_ps

    peer_enterprise_value = pick_numeric(peer_key_metrics, "enterpriseValueTTM", "enterpriseValue")
    if pd.isna(peer_enterprise_value) and pd.notna(peer_market_cap):
        peer_enterprise_value = peer_market_cap + peer_net_debt

    peer_ebitda_ttm = pick_numeric(peer_key_metrics, "ebitdaTTM", "ebitda")
    peer_ev_to_ebitda = pick_numeric(
        peer_key_metrics,
        "enterpriseValueOverEBITDATTM",
        "enterpriseValueOverEBITDA",
    )
    if pd.isna(peer_ebitda_ttm) and pd.notna(peer_enterprise_value) and pd.notna(peer_ev_to_ebitda) and peer_ev_to_ebitda > 0:
        peer_ebitda_ttm = peer_enterprise_value / peer_ev_to_ebitda

    peer_ebit_ttm = pick_numeric(peer_key_metrics, "ebitTTM", "ebit")
    peer_ev_to_ebit = pick_numeric(
        peer_key_metrics,
        "enterpriseValueOverEBITTTM",
        "enterpriseValueOverEBIT",
    )
    if pd.isna(peer_ebit_ttm) and pd.notna(peer_enterprise_value) and pd.notna(peer_ev_to_ebit) and peer_ev_to_ebit > 0:
        peer_ebit_ttm = peer_enterprise_value / peer_ev_to_ebit

    peer_ev_to_revenue = pick_numeric(
        peer_key_metrics,
        "enterpriseValueOverRevenueTTM",
        "enterpriseValueOverRevenue",
    )
    if pd.isna(peer_ev_to_revenue) and pd.notna(peer_enterprise_value) and pd.notna(peer_revenue_ttm) and peer_revenue_ttm > 0:
        peer_ev_to_revenue = peer_enterprise_value / peer_revenue_ttm
    if pd.isna(peer_ev_to_ebitda) and pd.notna(peer_enterprise_value) and pd.notna(peer_ebitda_ttm) and peer_ebitda_ttm > 0:
        peer_ev_to_ebitda = peer_enterprise_value / peer_ebitda_ttm
    if pd.isna(peer_ev_to_ebit) and pd.notna(peer_enterprise_value) and pd.notna(peer_ebit_ttm) and peer_ebit_ttm > 0:
        peer_ev_to_ebit = peer_enterprise_value / peer_ebit_ttm

    peer_metric_rows.append(
        {
            "symbol": peer_symbol,
            "companyName": peer_quote.get("name") or peer_profile.get("companyName") or peer_symbol,
            "price": peer_price,
            "marketCap": peer_market_cap,
            "enterpriseValue": peer_enterprise_value,
            "netDebt": peer_net_debt,
            "peRatio": peer_pe,
            "priceToSales": peer_ps,
            "priceToBook": peer_pb,
            "priceToFreeCashFlow": peer_pfcf,
            "evToRevenue": peer_ev_to_revenue,
            "evToEbitda": peer_ev_to_ebitda,
            "evToEbit": peer_ev_to_ebit,
            "epsTtm": peer_eps_ttm,
            "revenuePerShareTtm": peer_revenue_per_share_ttm,
            "bookValuePerShareTtm": peer_book_value_per_share_ttm,
            "freeCashFlowPerShareTtm": peer_free_cash_flow_per_share_ttm,
            "revenueTtm": peer_revenue_ttm,
            "ebitdaTtm": peer_ebitda_ttm,
            "ebitTtm": peer_ebit_ttm,
        }
    )

peer_market_multiples = pd.DataFrame(peer_metric_rows)
numeric_peer_columns = [
    "price",
    "marketCap",
    "enterpriseValue",
    "netDebt",
    "peRatio",
    "priceToSales",
    "priceToBook",
    "priceToFreeCashFlow",
    "evToRevenue",
    "evToEbitda",
    "evToEbit",
    "epsTtm",
    "revenuePerShareTtm",
    "bookValuePerShareTtm",
    "freeCashFlowPerShareTtm",
    "revenueTtm",
    "ebitdaTtm",
    "ebitTtm",
]
for column in numeric_peer_columns:
    if column in peer_market_multiples.columns:
        peer_market_multiples[column] = pd.to_numeric(peer_market_multiples[column], errors="coerce")

peer_market_multiples

## 7. Implied Price Table

This section converts both equity-value multiples and enterprise-value multiples into implied prices for the selected ticker. The equity-value methods use per-share drivers directly, while the EV methods estimate enterprise value first, then convert that to equity value per share after adjusting for net debt and share count.

In [ ]:
# 8. Build market-based implied prices
company_share_count = pd.to_numeric(company_market_inputs.at[0, "shareCount"], errors="coerce")
if (pd.isna(company_share_count) or company_share_count <= 0) and pd.notna(market_cap) and pd.notna(current_price) and current_price > 0:
    company_share_count = market_cap / current_price

company_net_debt = pd.to_numeric(company_market_inputs.at[0, "netDebt"], errors="coerce")
if pd.isna(company_net_debt):
    company_net_debt = 0.0

pricing_methods = [
    {
        "method": "P/E",
        "peerColumn": "peRatio",
        "driverLabel": "EPS (TTM)",
        "driverValue": company_market_inputs.at[0, "epsTtm"],
        "upperBound": 200,
        "valuationType": "equity",
    },
    {
        "method": "P/S",
        "peerColumn": "priceToSales",
        "driverLabel": "Revenue / Share (TTM)",
        "driverValue": company_market_inputs.at[0, "revenuePerShareTtm"],
        "upperBound": 50,
        "valuationType": "equity",
    },
    {
        "method": "P/B",
        "peerColumn": "priceToBook",
        "driverLabel": "Book Value / Share (TTM)",
        "driverValue": company_market_inputs.at[0, "bookValuePerShareTtm"],
        "upperBound": 80,
        "valuationType": "equity",
    },
    {
        "method": "P/FCF",
        "peerColumn": "priceToFreeCashFlow",
        "driverLabel": "Free Cash Flow / Share (TTM)",
        "driverValue": company_market_inputs.at[0, "freeCashFlowPerShareTtm"],
        "upperBound": 250,
        "valuationType": "equity",
    },
    {
        "method": "EV / Revenue",
        "peerColumn": "evToRevenue",
        "driverLabel": "Revenue (TTM)",
        "driverValue": company_market_inputs.at[0, "revenueTtm"],
        "upperBound": 30,
        "valuationType": "enterprise",
    },
    {
        "method": "EV / EBITDA",
        "peerColumn": "evToEbitda",
        "driverLabel": "EBITDA (TTM)",
        "driverValue": company_market_inputs.at[0, "ebitdaTtm"],
        "upperBound": 80,
        "valuationType": "enterprise",
    },
    {
        "method": "EV / EBIT",
        "peerColumn": "evToEbit",
        "driverLabel": "EBIT (TTM)",
        "driverValue": company_market_inputs.at[0, "ebitTtm"],
        "upperBound": 100,
        "valuationType": "enterprise",
    },
]

market_based_rows = []
method_diagnostics = []
expected_columns = [
    "Method",
    "Valuation Type",
    "Driver",
    "Driver Value",
    "Peer Sample Size",
    "Peer Multiple 25%",
    "Peer Multiple Median",
    "Peer Multiple 75%",
    "Implied Price Low",
    "Implied Price Median",
    "Implied Price High",
    "Upside / Downside To Current",
]

for method in pricing_methods:
    peer_series = pd.to_numeric(peer_market_multiples[method["peerColumn"]], errors="coerce").dropna()
    peer_series = peer_series[(peer_series > 0) & (peer_series <= method["upperBound"])]
    driver_value = pd.to_numeric(method["driverValue"], errors="coerce")

    status = "usable"
    reason = ""
    if peer_series.empty:
        status = "excluded"
        reason = "No peer multiples survived the positive-value and upper-bound filters."
    elif pd.isna(driver_value):
        status = "excluded"
        reason = "The company's valuation driver is missing."
    elif driver_value <= 0:
        status = "excluded"
        reason = "The company's valuation driver is non-positive."
    elif method["valuationType"] == "enterprise" and (pd.isna(company_share_count) or company_share_count <= 0):
        status = "excluded"
        reason = "Share count is missing, so enterprise value cannot be converted into equity value per share."

    method_diagnostics.append(
        {
            "Method": method["method"],
            "Valuation Type": method["valuationType"],
            "Driver": method["driverLabel"],
            "Driver Value": driver_value,
            "Peer Column": method["peerColumn"],
            "Peer Sample Size": int(peer_series.count()),
            "Status": status,
            "Reason": reason or "Included in implied price output.",
        }
    )

    if status != "usable":
        continue

    peer_multiple_low = float(peer_series.quantile(0.25))
    peer_multiple_median = float(peer_series.median())
    peer_multiple_high = float(peer_series.quantile(0.75))

    if method["valuationType"] == "equity":
        implied_low = peer_multiple_low * float(driver_value)
        implied_median = peer_multiple_median * float(driver_value)
        implied_high = peer_multiple_high * float(driver_value)
    else:
        enterprise_value_low = peer_multiple_low * float(driver_value)
        enterprise_value_median = peer_multiple_median * float(driver_value)
        enterprise_value_high = peer_multiple_high * float(driver_value)
        implied_low = (enterprise_value_low - company_net_debt) / company_share_count
        implied_median = (enterprise_value_median - company_net_debt) / company_share_count
        implied_high = (enterprise_value_high - company_net_debt) / company_share_count

    market_based_rows.append(
        {
            "Method": method["method"],
            "Valuation Type": method["valuationType"].title(),
            "Driver": method["driverLabel"],
            "Driver Value": float(driver_value),
            "Peer Sample Size": int(peer_series.count()),
            "Peer Multiple 25%": peer_multiple_low,
            "Peer Multiple Median": peer_multiple_median,
            "Peer Multiple 75%": peer_multiple_high,
            "Implied Price Low": float(implied_low),
            "Implied Price Median": float(implied_median),
            "Implied Price High": float(implied_high),
            "Upside / Downside To Current": float(implied_median) / current_price - 1 if pd.notna(current_price) and current_price else pd.NA,
        }
    )

market_pricing_method_diagnostics = pd.DataFrame(method_diagnostics)
market_based_pricing = pd.DataFrame(market_based_rows, columns=expected_columns)
if not market_based_pricing.empty:
    market_based_pricing = market_based_pricing.sort_values("Implied Price Median").reset_index(drop=True)

if market_based_pricing.empty:
    print(
        f"No usable market-based pricing methods were available for {SYMBOL}. "
        "Review the method diagnostics below to see which drivers or peer multiples were excluded."
    )
    market_pricing_method_diagnostics
else:
    market_based_pricing

In [ ]:
# 9. Plot market-based implied prices
if market_based_pricing.empty:
    print("No implied price chart was generated because Cell 8 did not produce any usable pricing methods.")
else:
    market_pricing_fig = go.Figure()
    market_pricing_fig.add_trace(
        go.Bar(
            x=market_based_pricing["Method"],
            y=market_based_pricing["Implied Price Median"],
            name="Peer median implied price",
            marker_color="#38bdf8",
            customdata=market_based_pricing[["Implied Price Low", "Implied Price High"]],
            hovertemplate=(
                "Method: %{x}<br>Median implied price: %{y:.2f}<br>"
                "Low-high range: %{customdata[0]:.2f} to %{customdata[1]:.2f}<extra></extra>"
            ),
        )
    )

    if pd.notna(current_price):
        market_pricing_fig.add_hline(
            y=current_price,
            line_dash="dash",
            line_color="#f97316",
            annotation_text=f"Current price: {current_price:.2f}",
            annotation_position="top left",
        )

    apply_standard_figure_layout(
        market_pricing_fig,
        title=f"{chart_label} market-based implied pricing",
        yaxis_title="USD per share",
    )
    market_pricing_fig.show(config={"responsive": True, "displaylogo": False})